# Part.6 Pencarian Dokumen


**RAHMA** **NURHALIZA** (**210411100176**)

**PPW** **A**

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import re
from IPython.display import display, Markdown

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/PPW/tugas/preprocessing-cnnnews.csv")
df.head()

,judul,isi,tanggal,kategori,berita_clean,case_folding,tokenize,stopword_removal
0,Jokowi Groundbreaking 2 Investasi Asing di IKN...,Presiden Joko Widodo (Jokowi) dijadwalkan mela...,"Selasa, 10 Sep 2024 20:02 WIB",Ekonomi,Presiden Joko Widodo Jokowi dijadwalkan melaku...,presiden joko widodo jokowi dijadwalkan melaku...,"['presiden', 'joko', 'widodo', 'jokowi', 'dija...",presiden joko widodo jokowi dijadwalkan peleta...
1,Amran Copot Direktur Diduga Calo Pengadaan Bar...,Menteri Pertanian Andi Amran Sulaiman mencopot...,"Selasa, 10 Sep 2024 19:11 WIB",Ekonomi,Menteri Pertanian Andi Amran Sulaiman mencopot...,menteri pertanian andi amran sulaiman mencopot...,"['menteri', 'pertanian', 'andi', 'amran', 'sul...",menteri pertanian andi amran sulaiman mencopot...
2,Bahlil Bicara Pasokan Listrik di Tengah Rencan...,Menteri Energi dan Sumber Daya Mineral (ESDM) ...,"Selasa, 10 Sep 2024 18:32 WIB",Ekonomi,Menteri Energi dan Sumber Daya Mineral ESDM Ba...,menteri energi dan sumber daya mineral esdm ba...,"['menteri', 'energi', 'dan', 'sumber', 'daya',...",menteri energi sumber daya mineral esdm bahlil...
3,Pertamina Kaji Ubah Minyak Jelantah Jadi Avtur,PT Pertamina (Persero) mengkaji pengembangan b...,"Selasa, 10 Sep 2024 17:43 WIB",Ekonomi,PT Pertamina Persero mengkaji pengembangan bah...,pt pertamina persero mengkaji pengembangan bah...,"['pt', 'pertamina', 'persero', 'mengkaji', 'pe...",pt pertamina persero mengkaji pengembangan bah...
4,"Dukung PMI, Bank Mandiri Perluas Akses Livin' ...",Bank Mandiri menghadirkan program Livin' Aroun...,"Selasa, 10 Sep 2024 17:18 WIB",Ekonomi,Bank Mandiri menghadirkan program Livin Around...,bank mandiri menghadirkan program livin around...,"['bank', 'mandiri', 'menghadirkan', 'program',...",bank mandiri menghadirkan program livin around...


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['isi'], df['kategori'], test_size=0.2, random_state=42)

## TF-IDF

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['stopword_removal'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_df.head(200)

,abad,abdul,abdulhadi,abinisa,absen,absjal,absrhr,abu,acara,aceh,...,yudo,yugen,yuk,yulianto,zaenal,zayana,zero,zona,zulhas,zulkifli
0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.024308,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.088183,0.000000,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.047426,0.000000,0.000000,0.0,0.0
96,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0
97,0.0,0.0,0.0,0.0,0.0,0.0,0.106948,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0
98,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0


## Reduksi Dimensi dengan  Truncated SVD

In [ ]:
svd = TruncatedSVD(n_components=100, random_state=42)  # Mulai dengan 100 komponen, bisa dikurangi lagi nanti
tfidf_svd_matrix = svd.fit_transform(tfidf_matrix)
svd_df = pd.DataFrame(tfidf_svd_matrix)
#Transpose matriks hasil SVD
#tfidf_svd_matrix_transposed = tfidf_svd_matrix.T
svd_df.head(100)
#svd_df.shape

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.064586,0.188992,0.008672,-0.027832,-0.006030,-0.147161,0.286516,0.018550,0.119475,-0.008067,...,0.011600,0.019177,0.015856,-0.004680,-0.016401,0.002696,0.000870,0.001023,-0.000633,0.001782
1,0.016457,0.179865,-0.013202,0.021377,0.077344,0.031816,0.088990,0.029517,0.107614,-0.129625,...,0.003997,-0.028164,0.000970,-0.006774,0.002675,-0.000255,-0.001913,0.002492,-0.001593,-0.002080
2,0.031062,0.144795,-0.010155,-0.003804,0.010489,-0.027065,0.007119,0.199850,-0.069954,-0.003941,...,-0.005434,-0.002698,-0.001227,0.000231,-0.000904,0.000405,-0.000557,-0.000371,-0.000362,-0.000086
3,0.047709,0.195778,-0.012764,-0.003038,-0.012011,-0.085911,-0.055248,0.161753,-0.076000,0.030667,...,0.001599,0.009062,-0.009052,-0.002909,0.002805,-0.001042,0.001103,0.001390,-0.001164,-0.000268
4,0.052167,0.133710,-0.009733,-0.010554,0.015350,-0.009544,0.048062,0.100011,0.037437,-0.004073,...,-0.008579,-0.011129,-0.000627,0.006025,-0.005575,0.000272,-0.000378,0.000010,0.001317,0.000176
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.114567,0.043904,0.010629,0.000050,0.042504,-0.057286,0.129163,-0.039097,-0.084506,-0.011177,...,-0.001750,0.002290,0.002323,-0.003260,0.002104,0.001595,0.001329,0.000764,-0.000653,-0.001204
96,0.154076,0.024865,0.043760,0.007618,-0.013868,0.006646,0.006122,-0.016914,-0.068085,-0.151474,...,0.000436,0.009560,0.014421,-0.007335,-0.000831,0.009769,-0.004656,0.001117,-0.002290,0.001260
97,0.559221,-0.018326,-0.094225,-0.064428,0.029930,-0.003415,-0.002151,-0.018271,-0.023394,-0.141488,...,-0.004422,0.000370,-0.025488,-0.045139,-0.005575,0.011777,0.016205,0.003124,0.006595,0.009041
98,0.545225,-0.022100,-0.124938,-0.021011,0.013938,-0.005100,0.013414,-0.001562,-0.017073,-0.213457,...,0.003371,-0.006210,-0.028595,-0.019757,0.008564,0.012005,0.002918,-0.003305,-0.009059,0.006263


## Transform query ke bentuk TF-IDF dan reduksi SVD

In [ ]:
#Transform query ke bentuk TF-IDF dan reduksi SVD
query = input("Masukkan kalimat yang ingin dicari: ")  # Ganti dengan kalimat yang ingin dicari
query_tfidf = vectorizer.transform([query])
query_svd = svd.transform(query_tfidf)
query_df = pd.DataFrame(query_svd)
#query_df

Masukkan kalimat yang ingin dicari: indonesia vs australia


## Hitung kemiripan (cosine similarity) antara query dan dokumen di dimensi yang lebih rendah

In [ ]:
cosine_similarities = cosine_similarity(query_svd, tfidf_svd_matrix).flatten()

In [ ]:
similarity_threshold = 0.5
related_docs_indices = [i for i, sim in enumerate(cosine_similarities) if sim > similarity_threshold]
related_documents = df.iloc[related_docs_indices].copy()
related_documents['similarity'] = cosine_similarities[related_docs_indices]  # Tambahkan kolom similarity

print("kalimat yang dicari:",query)

# Menampilkan dokumen terkait dengan format yang diinginkan
print("=" * 50)
print(f"{'Dokumen Terkait':^50}")
print("=" * 50)
displayed_count = 0

for index, row in related_documents.iterrows():
    print(f"Dokumen #{displayed_count + 1}")
    print(f"Judul     : {row['judul']}")
    print(f"Tanggal   : {row['tanggal']}")
    print(f"Kategori  : {row['kategori']}")
    print(f"Similarity: {row['similarity']:.4f}")
    print(f"Isi Berita:\n{row['isi']}")
    print("-" * 50)  # Pemisah antar dokumen
    displayed_count += 1

# Menampilkan jumlah dokumen yang ditampilkan
print(f"\nJumlah dokumen yang ditampilkan: {displayed_count}")
print("=" * 50)

kalimat yang dicari: indonesia vs australia
                 Dokumen Terkait                  
Dokumen #1
Judul     : Link Live Streaming Indonesia vs Australia di Kualifikasi Piala Dunia
Tanggal   : Selasa, 10 Sep 2024 17:04 WIB
Kategori  : Olahraga
Similarity: 0.5799
Isi Berita:
Link live streaming Timnas Indonesia vs Australia di putaran ketiga Kualifikasi Piala Dunia 2026 Zona Asia dapat dilihat dari artikel berikut. Timnas Indonesia akan menjalani laga kedua di Grup C Kualifikasi Piala Dunia 2026 Zona Asia dengan menghadapi Australia. Laga ini akan berlangsung hari ini Selasa (10/9) malam WIB di Stadion Utama Gelora Bung Karno, Jakarta. Jadwal Siaran langsung Timnas Indonesia vs Australia akan disiarkan RCTI pada Selasa (10/9) pukul 19.00 WIB. Selain itu, laga kandang pertama Timnas Indonesia pada fase ketiga Kualifikasi Piala Dunia 2026 ini juga dapat disaksikan melalui live streaming di Vision+. Pilihan Redaksi Jadwal Siaran Langsung Timnas Indonesia vs Australia Prediksi Timnas